# bsc_02 — MVP Stage 1

Gate 1 đã **PROCEED**. Chạy plan §6 bước 3→13 theo ma trận chuẩn hóa S/D/I/H.

**Notebook này cố ý MỎNG.** Mọi logic nằm ở `bsc/mvp.py` + `bsc/experiment.py` +
`bsc/atlas.py`, có test chạy trên phantom (`tests/test_mvp.py`, 11 test). Trước đây logic
nằm thẳng trong notebook nên chỉ lộ lỗi khi chạy trên Colab — sửa cách đó rồi.

| Trục | |
|---|---|
| **S** | S0 xương GT · S1 GT+jitter · S2 xương dự đoán |
| **D** | D0 oracle · D1 atlas theo fold |
| **I** | I0 mri · I1 +grad · I2 +sdf · I3 +prob |
| **H** | H0 chỉ occupancy · H1 +presence |

`P0=S0D0I0H0` · `P1=S0D0I1H0` · `P2=S0D1I2H1` · `P3=S0D1I3H1` (**P3 = M8-D**, một run)

**Thiết kế đánh giá:** train/val từ 404 ca TRAIN, fold theo `splits_zib_v1_fixed.json`
(train = fold 1–4, val = fold 0). Baseline = prediction **out-of-fold** 150ep.
**103 ca test không đụng tới.** Atlas D1 chỉ xây từ ca train của fold.

**Báo cáo (M0 §6):** không dùng ASSD tổng — mục tiêu quy về ASSD tổng chỉ 0.006mm, nhỏ
ngang sai khác giữa hai implementation metric. Dùng **mẫu số vùng mỏng** + presence F1.


### 0. Config + kiểm setup

In [1]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL, REPO_DIR = "https://github.com/AIVIETNAM-AIO-Tuan/bsCart-net.git", "/content/repo"
import os, sys, glob, json
if not os.path.isdir(f"{REPO_DIR}/bsc"):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
!pip install -q nibabel SimpleITK 2>/dev/null

# nap lai bsc sau git pull -> khoi restart kernel khi module thay doi
for _m in [k for k in list(sys.modules) if k == "bsc" or k.startswith("bsc.")]:
    del sys.modules[_m]

import numpy as np
from tqdm import tqdm
from bsc import core, metrics, headroom, io_utils, atlas as atlas_mod, mvp
from bsc import experiment as X
from bsc.core import RayConfig

# TU TIM duong dan - Drive hay bi sap xep lai (nnUNet_raw da chuyen vao OAI_seg/).
# Hardcode lam cell no IndexError kho hieu o cho khac.
D = "/content/drive/MyDrive"

def _find_dir(name, hints=()):
    for h in hints:
        if os.path.isdir(h):
            return h
    for pat in (f"{D}/{name}", f"{D}/*/{name}", f"{D}/*/*/{name}"):
        for d in sorted(glob.glob(pat)):
            if os.path.isdir(d):
                return d
    return None

RAW_ROOT = _find_dir("nnUNet_raw", [f"{D}/OAI_seg/nnUNet_raw", f"{D}/nnUNet_raw"])
BSC_ROOT = _find_dir("bsc", [f"{D}/OAI_seg/bsc", f"{D}/bsc"])
assert RAW_ROOT, f"Khong tim thay nnUNet_raw duoi {D}. Kiem lai Drive."
assert BSC_ROOT, f"Khong tim thay thu muc bsc duoi {D}."
RAW = f"{RAW_ROOT}/Dataset001_KneeOA"
PROB_DIR = f"{BSC_ROOT}/baselines/oof_prob"
print("RAW      =", RAW)
print("BSC_ROOT =", BSC_ROOT)
for sub in ["splits", "baselines", "atlas", "runs"]:
    os.makedirs(f"{BSC_ROOT}/{sub}", exist_ok=True)

cfg = RayConfig()
CART = {"femoral_cart": 2, "med_tib_cart": 4}
BONE = {"femoral_cart": 1, "med_tib_cart": 3}

ok = True
def chk(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"{'OK   ' if cond else 'THIEU'} | {name}{('  -> ' + detail) if detail else ''}")

labs = sorted(glob.glob(f"{RAW}/labelsTr/oaizib_*.nii.gz"))
chk("imagesTr", len(glob.glob(f"{RAW}/imagesTr/*_0000.nii.gz")) > 0)
chk("labelsTr", len(labs) > 0, f"{len(labs)} nhan")

SP = None
if labs:
    g0, SP = io_utils.load_nii(labs[0])
    SP = tuple(float(x) for x in SP)
    chk("nhan co xuong+sun", {1,2,3,4}.issubset(set(int(v) for v in np.unique(g0))))
    same = np.allclose(SP, core.SPACING, atol=1e-3)
    chk("spacing", True, f"{tuple(round(x,4) for x in SP)}"
        + ("" if same else f"  (khac core.SPACING {core.SPACING} - da xu ly)"))
    bad = []
    for p in labs[::40]:
        _, s = io_utils.load_nii(p)
        if not np.allclose(s, SP, atol=1e-3):
            bad.append(os.path.basename(p))
    chk("spacing dong nhat", not bad, f"kiem {len(labs[::40])} ca")

SPLITS = f"{BSC_ROOT}/splits/splits_zib_v1_fixed.json"
chk("splits_zib_v1_fixed", os.path.exists(SPLITS))
CVP = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_150epochs*/", recursive=True)
chk("CV 150ep (baseline OOF)", bool(CVP) and
    len(glob.glob(f"{CVP[0]}/fold_*/validation/oaizib_*.nii.gz")) > 0)

n_prob = len(glob.glob(f"{PROB_DIR}/**/*.nii.gz", recursive=True))
print(f"(tuy chon) OOF softmax cho P3/M8-D -> {n_prob} file"
      + ("" if n_prob else "   (chua co: P0-P2 van chay duoc)"))
print()
print("=> SAN SANG" if ok else "=> THIEU NGUYEN LIEU - dung lai")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 41.1 MB/s eta 0:00:00
OK    | imagesTr
OK    | labelsTr  -> 404 nhan
OK    | nhan co xuong+sun
OK    | spacing  -> (0.3646, 0.3646, 0.7)  (khac core.SPACING (0.7, 0.3646, 0.3646) - da xu ly)
OK    | spacing dong nhat  -> kiem 11 ca
OK    | splits_zib_v1_fixed
OK    | CV 150ep (baseline OOF)
(tuy chon) OOF softmax cho P3/M8-D -> 0 file   (chua co: P0-P2 van chay duoc)

=> SAN SANG


### 1. QC hình học trên xương thật — §6 bước 3, 5, 6

**Cổng dừng:** M3 < 99.5% · M4 < 90% · M2 ASSD > 0.1mm · khung suy biến > 0%.

In [2]:

QC_CKPT, N_QC = f"{BSC_ROOT}/runs/mvp_geom_qc.jsonl", 10
cases_all = sorted(os.path.basename(p)[:-len("_0000.nii.gz")]
                   for p in glob.glob(f"{RAW}/imagesTr/*_0000.nii.gz"))
cases_all = [c for c in cases_all if c.startswith("oaizib_")]

done = ({(json.loads(l)["case"], json.loads(l)["cls"]) for l in open(QC_CKPT)}
        if os.path.exists(QC_CKPT) else set())

# Nguon rieng cho tung lop (QC chay ca hai lop)
def _src_for(cls):
    return mvp.NiftiCaseSource(RAW, cls, CART[cls], BONE[cls], lambda c: None)

with open(QC_CKPT, "a") as fh:
    for cid in tqdm(cases_all[:N_QC], desc="geom QC"):
        gt, _ = io_utils.load_nii(f"{RAW}/labelsTr/{cid}.nii.gz")
        for c in CART:
            if (cid, c) in done: continue
            bone, cart = (gt == BONE[c]), (gt == CART[c])
            if not bone.any() or not cart.any(): continue
            row = mvp.geometry_qc_case(_src_for(c), cid, c, bone, cart, cfg)
            fh.write(json.dumps(row) + chr(10)); fh.flush()

rows = [json.loads(l) for l in open(QC_CKPT)]
print()
print(f"{'lop':<14}{'M3':>8}{'M4':>8}{'M2 Dice':>10}{'M2 ASSD':>10}{'suy bien':>11}{'cong':>7}")
for c in CART:
    s_ = mvp.summarize_geometry_qc(rows, c)
    if s_ is None:
        print(f"{c:<14}  (chua co ban ghi hop le)"); continue
    g_ = mvp.geometry_qc_gate(s_)
    print(f"{c:<14}{s_['m3_normals_ok']:>7.1%}{s_['m4_single_interval']:>8.1%}"
          f"{s_['m2_dice']:>10.3f}{s_['m2_assd_mm']:>9.3f}mm"
          f"{s_['frame_degenerate']:>10.0%}{'PASS' if g_['pass'] else 'FAIL':>7}")
    if not g_['pass']:
        print(f"   -> truot: {[k for k,v in g_.items() if k!='pass' and not v]}")

print()
print("CONG §3.5: M3>99.5% | M4>90% | M2 ASSD<0.1mm | khung suy bien=0%")

geom QC: 100%|██████████| 10/10 [00:03<00:00,  2.74it/s]


lop                 M3      M4   M2 Dice   M2 ASSD   suy bien   cong
femoral_cart    99.8%   99.8%     0.995    0.009mm        0%   PASS
med_tib_cart    99.6%   99.7%     0.993    0.008mm        0%   PASS

CONG §3.5: M3>99.5% | M4>90% | M2 ASSD<0.1mm | khung suy bien=0%


### 2. Split + atlas theo fold — §2.2

Atlas chỉ xây từ ca **train** của fold; `assert_no_leak` biến vi phạm §2.2 thành lỗi.
**P0/P1 dùng D0 (oracle) nên không cần atlas** — atlas chỉ cần từ P2.

In [3]:
CLS = "femoral_cart"       # §3.1: lop DAN de debug pipeline. Doi sang med_tib_cart o buoc 11.
N_TRAIN, N_VAL, RAYS = 40, 10, 20000

fold_of = json.load(open(SPLITS))["fold_of"]
zib = [c for c in cases_all if c in fold_of]
train_ids_all = [c for c in zib if fold_of[c] != 0]   # DAY DU - B3 dung
train_ids = train_ids_all[:N_TRAIN]
val_ids   = [c for c in zib if fold_of[c] == 0][:N_VAL]
print(f"train {len(train_ids)}/{len(train_ids_all)} kha dung | val {len(val_ids)}")

CV_DIR = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_150epochs*/", recursive=True)[0]
def baseline_path(cid):
    p = glob.glob(f"{CV_DIR}/fold_*/validation/{cid}.nii.gz")
    return p[0] if p else None

SRC = mvp.NiftiCaseSource(RAW, CLS, CART[CLS], BONE[CLS], baseline_path, PROB_DIR)

ATLAS_PATH = f"{BSC_ROOT}/atlas/atlas_{CLS}_fold0.npz"
if os.path.exists(ATLAS_PATH):
    z = np.load(ATLAS_PATH, allow_pickle=True)
    ATLAS = atlas_mod.ArticularAtlas(z["prob"], int(z["n_bins"]), float(z["lo"]),
                                     float(z["hi"]), tuple(z["case_ids"]), 0, None)
    print(f"Nap atlas: {len(ATLAS.case_ids)} ca")
else:
    def _al(cid): return SRC.mri(cid), SRC.bone_gt(cid), SRC.cart_gt(cid)
    ATLAS = atlas_mod.build_articular_atlas(train_ids, _al, SP, cfg, n_bins=24,
                                            min_count=3, fold=0)
    np.savez_compressed(ATLAS_PATH, prob=ATLAS.prob, n_bins=ATLAS.n_bins, lo=ATLAS.lo,
                        hi=ATLAS.hi, case_ids=np.array(ATLAS.case_ids))
    print(f"Xay atlas tu {len(ATLAS.case_ids)} ca -> {ATLAS_PATH}")

atlas_mod.assert_no_leak(ATLAS, val_ids)
print(f"assert_no_leak OK | phu {np.isfinite(ATLAS.prob).mean():.1%} o luoi, "
      f"P(khop) TB {np.nanmean(ATLAS.prob):.3f}")

train 40 | val 10
Nap atlas: 40 ca
assert_no_leak OK | phu 34.8% o luoi, P(khop) TB 0.354


In [4]:
# Atlas co PHAN BIET duoc khong, hay chi chon bua? Do tren ca VAL.
probs = ATLAS.prob[np.isfinite(ATLAS.prob)]
print(f"phan bo P(khop): p10 {np.percentile(probs,10):.2f} | median "
      f"{np.median(probs):.2f} | p90 {np.percentile(probs,90):.2f}")
print(f"ty le o co P>=0.8: {(probs>=0.8).mean():.1%} | P<=0.2: {(probs<=0.2).mean():.1%}")

print(f"\n{'ca':<14}{'node':>8}{'atlas chon':>12}{'oracle':>9}{'recall':>9}{'prec':>8}")
for cid in val_ids[:5]:
    bone, cart = SRC.bone_gt(cid), SRC.cart_gt(cid)
    _, verts, normals = core.bone_geometry(bone, SP, cfg)
    occ = core.occupancy_target(cart, verts, normals, cfg, SP)
    truth = core.oracle_domain(verts, occ, cfg)          # D0 - moc so sanh
    dom = atlas_mod.atlas_domain(verts, ATLAS, thr=0.2)  # D1
    rec = (dom & truth).sum() / max(truth.sum(), 1)
    pre = (dom & truth).sum() / max(dom.sum(), 1)
    print(f"{cid:<14}{len(verts):>8}{dom.mean():>11.1%}{truth.mean():>9.1%}"
          f"{rec:>9.1%}{pre:>8.1%}")


phan bo P(khop): p10 0.00 | median 0.00 | p90 1.00
ty le o co P>=0.8: 32.0% | P<=0.2: 61.6%

ca                node  atlas chon   oracle   recall    prec
oaizib_003      143389      41.1%    37.4%    99.2%   90.3%
oaizib_010      143083      41.7%    36.6%    99.6%   87.4%
oaizib_022      154945      40.9%    38.2%    98.2%   91.7%
oaizib_028      175331      40.2%    35.2%    99.8%   87.3%
oaizib_035      138192      40.7%    37.3%    99.4%   91.1%


In [5]:
# Xay TRUOC atlas med_tib de khoi ton GPU o buoc 11. KHONG doi CLS/SRC/ATLAS.
_cls2 = "med_tib_cart"
_path2 = f"{BSC_ROOT}/atlas/atlas_{_cls2}_fold0.npz"

if os.path.exists(_path2):
    print("da co:", _path2)
else:
    _src2 = mvp.NiftiCaseSource(RAW, _cls2, CART[_cls2], BONE[_cls2], baseline_path)
    def _al2(cid): return _src2.mri(cid), _src2.bone_gt(cid), _src2.cart_gt(cid)
    _a2 = atlas_mod.build_articular_atlas(train_ids, _al2, SP, cfg,
                                          n_bins=24, min_count=3, fold=0)
    np.savez_compressed(_path2, prob=_a2.prob, n_bins=_a2.n_bins, lo=_a2.lo,
                        hi=_a2.hi, case_ids=np.array(_a2.case_ids))
    atlas_mod.assert_no_leak(_a2, val_ids)
    print(f"Xay atlas {_cls2} tu {len(_a2.case_ids)} ca -> {_path2}")
    print(f"phu {np.isfinite(_a2.prob).mean():.1%} o luoi, "
          f"P(khop) TB {np.nanmean(_a2.prob):.3f}")

print("CLS van la:", CLS)   # phai in femoral_cart


da co: /content/drive/MyDrive/bsc/atlas/atlas_med_tib_cart_fold0.npz
CLS van la: femoral_cart


In [6]:
# Hai atlas co LAN vao nhau khong? Phep thu: dung atlas SAI LOP cho cung bo dinh.
# Neu file med_tib vo tinh chua du lieu femoral, no se khop vung femoral rat tot.
_p_fem = f"{BSC_ROOT}/atlas/atlas_femoral_cart_fold0.npz"
_p_tib = f"{BSC_ROOT}/atlas/atlas_med_tib_cart_fold0.npz"

def _load(p):
    z = np.load(p, allow_pickle=True)
    return atlas_mod.ArticularAtlas(z["prob"], int(z["n_bins"]), float(z["lo"]),
                                    float(z["hi"]), tuple(z["case_ids"]), 0, None)

A_FEM, A_TIB = _load(_p_fem), _load(_p_tib)

# 1. Hai mang xac suat phai KHAC nhau
same = np.array_equal(np.nan_to_num(A_FEM.prob, nan=-1),
                      np.nan_to_num(A_TIB.prob, nan=-1))
print(f"1) prob giong het nhau? {same}   <- phai la False")
print(f"   P TB: femoral {np.nanmean(A_FEM.prob):.3f} | med_tib {np.nanmean(A_TIB.prob):.3f}")
print(f"   cung tap ca train? {set(A_FEM.case_ids) == set(A_TIB.case_ids)}  <- True la dung")

# 2. Cheo: moi atlas phai khop lop CUA NO, va khop KEM lop kia
print(f"\n{'ca':<13}{'atlas':<9}{'tren xuong':<11}{'chon':>7}{'oracle':>8}{'recall':>8}")
for cid in val_ids[:3]:
    gt, _ = io_utils.load_nii(f"{RAW}/labelsTr/{cid}.nii.gz")
    for bone_name, bl, cl in [("femur", 1, 2), ("tibia", 3, 4)]:
        bone, cart = (gt == bl), (gt == cl)
        if not bone.any() or not cart.any(): continue
        _, verts, normals = core.bone_geometry(bone, SP, cfg)
        occ = core.occupancy_target(cart, verts, normals, cfg, SP)
        truth = core.oracle_domain(verts, occ, cfg)
        for aname, A in [("femoral", A_FEM), ("med_tib", A_TIB)]:
            dom = atlas_mod.atlas_domain(verts, A, thr=0.2)
            rec = (dom & truth).sum() / max(truth.sum(), 1)
            mark = "  <-- dung cap" if (aname == "femoral") == (bone_name == "femur") else ""
            print(f"{cid:<13}{aname:<9}{bone_name:<11}{dom.mean():>6.1%}"
                  f"{truth.mean():>8.1%}{rec:>8.1%}{mark}")

# 3. Trang thai phien hien tai
print(f"\n3) CLS = {CLS} | SRC lop = {SRC.cls} | ATLAS tu {len(ATLAS.case_ids)} ca")
print(f"   ATLAS dang nap co phai femoral? "
      f"{np.allclose(np.nan_to_num(ATLAS.prob,nan=-1), np.nan_to_num(A_FEM.prob,nan=-1))}")


1) prob giong het nhau? False   <- phai la False
   P TB: femoral 0.354 | med_tib 0.124
   cung tap ca train? True  <- True la dung

ca           atlas    tren xuong    chon  oracle  recall
oaizib_003   femoral  femur       41.1%   37.4%   99.2%  <-- dung cap
oaizib_003   med_tib  femur        3.8%   37.4%    0.0%
oaizib_003   femoral  tibia       15.4%   13.6%    0.0%
oaizib_003   med_tib  tibia       15.5%   13.6%   98.7%  <-- dung cap
oaizib_010   femoral  femur       41.7%   36.6%   99.6%  <-- dung cap
oaizib_010   med_tib  femur        3.3%   36.6%    0.0%
oaizib_010   femoral  tibia       16.1%   14.8%    0.0%
oaizib_010   med_tib  tibia       16.0%   14.8%   98.1%  <-- dung cap
oaizib_022   femoral  femur       40.9%   38.2%   98.2%  <-- dung cap
oaizib_022   med_tib  femur        3.0%   38.2%    0.0%
oaizib_022   femoral  tibia       14.3%   14.6%    0.0%
oaizib_022   med_tib  tibia       15.3%   14.6%   96.6%  <-- dung cap

3) CLS = femoral_cart | SRC lop = femoral_cart | ATLA

### 2b. Phase A — canonical P2 (checkpoint-locked engineering baseline)

Chạy P2 **config CŨ** (không đổi gì — §4.3), lưu checkpoint + hash, rồi **eval từ model
reload trên đĩa** (không dùng `net` trong RAM). Đây là `P2_oldcfg_engref` — mốc tham chiếu
sạch cho mọi revision Phase B. Xem `mapping_split_canonical_p2_decisions_vi.md` §4, §7.

⚠️ **KHÔNG chạy cell "CACHE HÌNH HỌC" ở mục 4** cho lần canonical này — để cache tắt cho
sạch (§2.6). Nếu đã lỡ chạy nó trong phiên, gỡ bằng: `mvp.build_case = mvp._orig_build_case`.

In [ ]:
# A1 - train P2 (config CU) + luu checkpoint + reload tu dia -------------------
import hashlib
from bsc import model as M

EPOCHS_A, LR_A, RAYS_A = 30, 3e-4, 20000
run_p2 = X.from_plan("P2", CLS, seed=1)                 # config CU, khong doi gi (§4.3)

# neu lo bat cache o phien nay thi go ra cho canonical
if getattr(mvp, "_orig_build_case", None) is not None:
    mvp.build_case = mvp._orig_build_case
    print("Da go geom cache cho canonical run.")

atlas_hash = hashlib.sha256(ATLAS.prob.tobytes()
                            + str(sorted(ATLAS.case_ids)).encode()).hexdigest()[:16]
split_hash = hashlib.sha256(json.dumps({"train": sorted(train_ids), "val": sorted(val_ids)},
                                       sort_keys=True).encode()).hexdigest()[:16]

res_p2 = mvp.train_run(run_p2, SRC, train_ids, val_ids, cfg, atlas=ATLAS,
                       rays_per_case=RAYS_A, epochs=EPOCHS_A, lr=LR_A, device="cuda")

run_dir = mvp.save_run(res_p2, BSC_ROOT, cfg, epochs=EPOCHS_A, lr=LR_A,
                       rays_per_case=RAYS_A, atlas_hash=atlas_hash, split_hash=split_hash,
                       dataset_revision="Dataset001_KneeOA", git_cwd=REPO_DIR)

# TU DAY moi eval dung MODEL RELOAD tu dia (khong dung res_p2.net trong RAM)
net = mvp.load_run_model(run_dir, device="cuda")
man = mvp.run_manifest(run_dir)
print("run_dir =", run_dir)
print(f"experiment_id: {man['experiment_id']}")
print(f"checkpoint_sha256: {man['checkpoint_sha256']} | git: {man['git_commit'][:8]} "
      f"| config_hash: {man['config_hash']}")
print(f"atlas_hash: {man['atlas_hash']} | split_hash: {man['split_manifest_hash']}")
print(f"val occ-Dice: {man['val_occ_dice']:.3f}")

In [ ]:
# A2 - §3.7 tu MODEL RELOAD, ghi vao run_dir (khong append file cu) --------------
EVAL_A = f"{run_dir}/eval_gate37.jsonl"
open(EVAL_A, "w").close()                               # thu muc moi -> file moi, sach
rows_a = []
for cid in tqdm(val_ids, desc="gate37@ckpt"):
    r = mvp.evaluate_case(run_p2, net, SRC, cid, cfg, ATLAS, device="cuda")
    if r:
        rows_a.append(r)
        open(EVAL_A, "a").write(json.dumps(r) + chr(10))

g = mvp.summarize_gate37(rows_a)
print(f"n={g['n']} | thin: B0 {g['thin_err_baseline_mm']:.4f} -> ray {g['thin_err_ray_mm']:.4f}mm"
      f" | rel {g['rel_improve']:+.1%} | {g['n_better']}/{g['n']} tot hon")
if "presence_f1" in g:
    print(f"presence F1 {g['presence_f1']:.3f} | absent recall {g['absent_recall']:.3f}")

json.dump({"gate37": g, "checkpoint_sha256": man["checkpoint_sha256"],
           "experiment_id": man["experiment_id"], "git_commit": man["git_commit"]},
          open(f"{run_dir}/gate37_summary.json", "w"), indent=2)
print("=> P2_oldcfg_engref khoa tai", run_dir)

In [ ]:
# A3 - Mapping audit A (node) vs B (nearest-ray), du 10 ca (§2.6) ----------------
aud = []
for c in val_ids:
    if not SRC.has_pred(c):
        continue                                       # can baseline pred lam surface points
    a = mvp.mapping_audit_case(SRC.cart_gt(c), SRC.baseline_cart(c),
                               SRC.bone_gt(c), SRC.spacing(c), cfg)
    if a:
        aud.append(a)

eA = np.array([a["thin_err_A_mm"] for a in aud])
eB = np.array([a["thin_err_B_mm"] for a in aud])
rel = abs(eB.mean() - eA.mean()) / eA.mean()
absA = sum(a["gt_vox_absent_A"] for a in aud)
absB = sum(a["gt_vox_absent_B"] for a in aud)

print(f"n = {len(aud)} ca")
print(f"thin err   Mapping A (node) {eA.mean():.4f}mm | Mapping B (ray) {eB.mean():.4f}mm"
      f" | doi {rel:.1%}")
print(f"GT voxel gan nham 'absent'  A {absA} -> B {absB}")
print(f"QC Mapping B: unassigned {np.mean([a['unassigned_frac_B'] for a in aud]):.1%}"
      f" | dist median {np.mean([a['distB_median_mm'] for a in aud]):.3f}mm"
      f" | p95 {np.mean([a['distB_p95_mm'] for a in aud]):.3f}mm"
      f" | max_dist {aud[0]['max_dist_mm']:.3f}mm")

json.dump(aud, open(f"{run_dir}/mapping_audit.json", "w"), indent=2)
print()
print("LUAT §2.6:  doi <5% => giu Mapping A + caveat  |  >10% => full provenance truoc final Gate")
print(f"=> {'GIU Mapping A' if rel < 0.05 else ('PROVENANCE truoc final Gate' if rel > 0.10 else 'VUNG XAM - xem ky per-case')}")

### 2c. Phase B — bước 1: micro-overfit đúng cấu hình S0-D1-H1

**Diagnostic trước khi revise** (review §11 step 1): xác nhận cấu hình **đang hỏng**
(S0-D1-H1) *có thể* thuộc lòng tia của 1 ca. Nếu occ-Dice train **không** đạt ~0.97–0.99
⇒ có **bug** ở input/target/kiến trúc/loss, **không phải** chuyện scale — dừng, tìm bug.
Nếu đạt ⇒ lỗi §3.7 là do generalization/sampling/loss ⇒ tiến hành revise (bước 2+).

Báo cáo **thin-ray recall** và **absent presence-acc** riêng — đó là chỗ §3.7 thất bại.

In [ ]:
# B1 - micro-overfit S0-D1-H1 (cau hinh dang HONG), tia phan tang -----------------
run_b = X.from_plan("P2", CLS, seed=1)          # S0-D1-I2-H1
rb = mvp.micro_overfit(run_b, SRC, train_ids[0], cfg, atlas=ATLAS,
                       rays=1200, epochs=150, lr=3e-3, stratify=True, device="cuda")
print(f"loss {rb['loss0']:.3f} -> {rb['lossN']:.3f}  | n_rays {rb['n_rays']} "
      f"| absent {rb['absent_frac']:.1%}")
print(f"occ-Dice(train) {rb['occ_dice']:.3f} @thr {rb['best_thr']:.2f}   (muc tieu 0.97-0.99)")
print(f"thin-ray recall {rb['thin_ray_recall']:.3f} | absent presence-acc {rb['absent_pres_acc']:.3f}")
print()
print("DOC:")
print("  occ-Dice >= 0.97  => config THUOC LONG duoc. Loi §3.7 = generalization/sampling/loss")
print("                       => revise (buoc 2: stratified full + loss). BUG loai tru.")
print("  occ-Dice <  0.90  => BUG o input/target/kien truc/loss - DUNG, tim bug, khong scale.")
print("  thin-recall / absent-acc thap  => chi ra dung cho revise nham toi.")

### 2d. Phase B — bước 2: SWEEP các lever (không cần push code)

B1 đã loại trừ bug (interior 2.69%, presence memorize 0.988) ⇒ lỗi §3.7 là **generalization
gap của presence**, không phải giới hạn biểu diễn. Bước 2 nhắm đúng chỗ đó.

**Mọi lever đã tham số hóa** — muốn thử ý tưởng mới thì **sửa `EXPERIMENTS` trong cell dưới
rồi chạy lại**, KHÔNG cần push code, KHÔNG cần restart:

| Lever | Giá trị | Nhắm vào |
|---|---|---|
| `stratify` | True/False | recall vùng mỏng 37% (review §8 sampling bias) |
| `absent_fp_weight` | 0 / 2 / 5 | **82% lỗi là FP absent** (§8.3) — thủ phạm chính |
| `per_ray_norm` | True/False | sụn dày chi phối gradient (§8) |
| `presence_gate` | hard/soft/none | hard-gate biến FP thành FN (§8.5) |
| `n_train` | 40 / nhiều hơn | generalization gap |

**Mốc so sánh:** P2 khóa **2.19mm** (Phase A) và B0 **0.5517mm**. Cổng Phase B: giảm mạnh từ
2.19 + absent FP giảm + thin recall tăng — chưa cần vượt B0 khi còn debug.

⚠️ Đây là **engineering sweep trên val 10 ca** — không phải final Gate. Cấu hình chọn ở đây
phải đánh giá lại trên split phân tầng ở Phase C (tránh selection bias).

In [ ]:
# B2 - SWEEP lever Phase B. Doi EXPERIMENTS roi chay lai - KHONG can push code.
from bsc import model as M

P2_LOCKED_MM, B0_MM = 2.1881, 0.5517          # moc Phase A (§15) va baseline

EXPERIMENTS = {
    # ten:            stratify, absent_fp_weight, per_ray_norm, presence_gate, n_train
    "baseline(P2)":   dict(stratify=False, absent_fp_weight=0.0, per_ray_norm=False,
                           presence_gate="hard", n_train=40),
    "stratify":       dict(stratify=True,  absent_fp_weight=0.0, per_ray_norm=False,
                           presence_gate="hard", n_train=40),
    "absentFP=3":     dict(stratify=True,  absent_fp_weight=3.0, per_ray_norm=False,
                           presence_gate="hard", n_train=40),
    "soft-gate":      dict(stratify=True,  absent_fp_weight=3.0, per_ray_norm=False,
                           presence_gate="soft", n_train=40),
}

EPOCHS_B, RAYS_B = 30, 20000
SWEEP = f"{BSC_ROOT}/runs/phaseB_sweep_{CLS}.jsonl"
run_b2 = X.from_plan("P2", CLS, seed=1)

print(f"{'cau hinh':<16}{'thin err':>10}{'vs P2':>9}{'vs B0':>9}{'tot/n':>8}{'presF1':>8}")
for name, e in EXPERIMENTS.items():
    tr = train_ids[:e["n_train"]]
    res = mvp.train_run(run_b2, SRC, tr, val_ids, cfg, atlas=ATLAS,
                        rays_per_case=RAYS_B, epochs=EPOCHS_B, device="cuda",
                        stratify=e["stratify"],
                        loss=M.LossWeights(absent_fp_weight=e["absent_fp_weight"],
                                           per_ray_norm=e["per_ray_norm"]))
    rows_e = [mvp.evaluate_case(run_b2, res.net, SRC, c, cfg, ATLAS, device="cuda",
                                presence_gate=e["presence_gate"]) for c in val_ids]
    g = mvp.summarize_gate37([r for r in rows_e if r])
    f1 = g.get("presence_f1", float("nan"))
    print(f"{name:<16}{g['thin_err_ray_mm']:>9.3f}mm"
          f"{g['thin_err_ray_mm']-P2_LOCKED_MM:>+9.3f}{g['thin_err_ray_mm']-B0_MM:>+9.3f}"
          f"{g['n_better']:>5}/{g['n']}{f1:>8.3f}")
    with open(SWEEP, "a") as fh:
        fh.write(json.dumps({"name": name, **e, "gate37": g,
                             "epochs": EPOCHS_B, "rays": RAYS_B}, default=str) + chr(10))

print()
print(f"moc: P2 khoa {P2_LOCKED_MM:.3f}mm | B0 {B0_MM:.3f}mm")
print("DOC: 'vs P2' AM = revision tot hon P2 cu. 'vs B0' AM = da vuot baseline (cong Stage 2).")
print(f"Da ghi {SWEEP}")

### 2e. Phase B — bước 3: PHÉP THỬ QUYẾT ĐỊNH (soft-gate + nhiều ca)

**Sweep B2 đã bác bỏ giả thuyết "loss/sampling bias là gốc rễ"** (review §8): `stratify` làm
*tệ hơn* (+0.05), `absentFP=3` gần như vô hiệu (−0.24, cùng cỡ nhiễu run-to-run ~0.25mm).
Chỉ `soft-gate` có tác dụng (−0.53) nhưng nó chỉ đổi cách reconstruct, không sửa model.
**0/10 ở mọi cấu hình.**

Lever duy nhất chưa thử nhắm đúng chẩn đoán B1 (presence memorize train 0.988 nhưng ảo giác
trên val = **generalization gap**): **nhiều dữ liệu hơn**. Hiện mới dùng 40/320 ca.

**Cổng ghi TRƯỚC khi chạy (tránh dời cột gôn):**

| Kết quả | Kết luận | Hành động |
|---|---|---|
| thin err **≲1.0mm** | thiếu dữ liệu là thủ phạm | scale tiếp — REVISE có cơ sở |
| thin err **~1.6–1.9mm** | 4× dữ liệu không giúp ⇒ **không phải** generalization gap | **DỪNG Phase B**, viết kết quả âm tính |

Cơ sở dừng: plan §3.7 — *"nếu model xương-GT không vượt ResEnc, dừng hoặc sửa giả thuyết"*.
Ta đã dùng xương GT (oracle), đã loại trừ bug (B1), đã biết trần biểu diễn tốt (0.026mm).
Nếu vẫn không đạt thì đó là **phát hiện khoa học**, không phải thất bại kỹ thuật.

In [ ]:
# B3 - PHEP THU QUYET DINH: giu lever CO tac dung (soft-gate), tang du lieu 4x.
# Bo stratify (lam TE hon) va absentFP (vo hieu) - B2 da bac bo chung.
from bsc import model as M

N_TRAIN_B3, EPOCHS_B3, RAYS_B3 = 160, 40, 20000
P2_LOCKED_MM, B0_MM = 2.1881, 0.5517

tr3 = train_ids_all[:N_TRAIN_B3] if "train_ids_all" in dir() else train_ids[:N_TRAIN_B3]
print(f"train {len(tr3)} ca (truoc: 40) | val {len(val_ids)} | epochs {EPOCHS_B3}")
if len(tr3) < N_TRAIN_B3:
    print(f"CANH BAO: chi co {len(tr3)} ca - dat N_TRAIN o muc 2 len >= {N_TRAIN_B3}")

run_b3 = X.from_plan("P2", CLS, seed=1)
res3 = mvp.train_run(run_b3, SRC, tr3, val_ids, cfg, atlas=ATLAS,
                     rays_per_case=RAYS_B3, epochs=EPOCHS_B3, device="cuda",
                     stratify=False, loss=M.LossWeights())

rows3 = [mvp.evaluate_case(run_b3, res3.net, SRC, c, cfg, ATLAS, device="cuda",
                           presence_gate="soft") for c in val_ids]
g3 = mvp.summarize_gate37([r for r in rows3 if r])

print()
print(f"thin err  {g3['thin_err_ray_mm']:.3f}mm"
      f"   vs P2 {g3['thin_err_ray_mm']-P2_LOCKED_MM:+.3f}"
      f"   vs B0 {g3['thin_err_ray_mm']-B0_MM:+.3f}"
      f"   tot {g3['n_better']}/{g3['n']}")
if "presence_f1" in g3:
    print(f"presence F1 {g3['presence_f1']:.3f} | absent recall {g3['absent_recall']:.3f}")

verdict = ("THIEU DU LIEU la thu pham -> scale tiep, REVISE co co so"
           if g3["thin_err_ray_mm"] <= 1.0 else
           "4x du lieu KHONG giup -> DUNG Phase B, viet ket qua AM TINH")
print()
print("=> " + verdict)

json.dump({"n_train": len(tr3), "epochs": EPOCHS_B3, "presence_gate": "soft",
           "gate37": g3, "verdict": verdict, "vs_P2": g3["thin_err_ray_mm"]-P2_LOCKED_MM,
           "vs_B0": g3["thin_err_ray_mm"]-B0_MM},
          open(f"{BSC_ROOT}/runs/phaseB_B3_{CLS}.json", "w"), indent=2, default=str)
print(f"Da ghi {BSC_ROOT}/runs/phaseB_B3_{CLS}.json")

### 3. M5 — tiny-set overfitting trên dữ liệu thật (§3.5, bước 8)

Trượt ⇒ có **bug** ở input/target/kiến trúc/loss. Đừng chỉnh hyperparameter.

In [7]:
from bsc import model as M
r5 = X.from_plan("P1", CLS, seed=1)
Xa, oa, pa = mvp.build_dataset(r5, SRC, train_ids[:3], cfg, rays_per_case=2000, seed=0)
i = np.random.default_rng(0).choice(len(Xa), min(4000, len(Xa)), replace=False)

net5 = M.RayEncoder1D(in_channels=len(r5.channels), with_presence=r5.with_presence)
h5 = M.fit(net5, Xa[i], oa[i], pa[i], epochs=80, batch_size=512, lr=3e-3, seed=0)
op, _ = M.predict_rays(net5, Xa[i])
m5_dice = float(2*((op>0.5) & oa[i].astype(bool)).sum()/((op>0.5).sum()+oa[i].sum()+1e-8))
print(f"M5 loss {h5[0]['loss']:.4f} -> {h5[-1]['loss']:.4f} | occ-Dice(train) {m5_dice:.3f}")
print("CONG: Dice > 0.90")

M5 loss 0.4567 -> 0.1051 | occ-Dice(train) 0.883
CONG: Dice > 0.90


In [8]:
# M5 truot 0.007 - do nguong, do luong tu hoa bien, hay do underfit that?
op, _ = M.predict_rays(net5, Xa[i])
tgt = oa[i].astype(bool)

# (1) Nguong 0.5 co phai cho tot nhat khong?
print("nguong  Dice")
best = (0, 0)
for t in np.arange(0.2, 0.81, 0.05):
    d = 2*((op > t) & tgt).sum() / ((op > t).sum() + tgt.sum() + 1e-8)
    best = max(best, (d, t))
    print(f"  {t:.2f}  {d:.3f}" + ("  <-- 0.5 mac dinh" if abs(t-0.5) < 1e-9 else ""))
print(f"=> tot nhat {best[0]:.3f} tai nguong {best[1]:.2f}")

# (2) Loi nam o O BIEN hay o TRONG LONG sun?
pred = op > best[1]
edge = np.zeros_like(tgt)
edge[:, 1:] |= tgt[:, 1:] != tgt[:, :-1]      # o ke chuyen tiep 0<->1
edge[:, :-1] |= tgt[:, 1:] != tgt[:, :-1]
err = pred != tgt
print(f"\nloi tong {err.mean():.2%} | o BIEN {err[edge].mean():.2%} "
      f"| trong LONG {err[~edge].mean():.2%}")
print(f"ty trong loi nam o bien: {err[edge].sum()/max(err.sum(),1):.1%}")

# (3) Loss con dang giam khong (underfit)?
last = [h["loss"] for h in h5[-10:]]
print(f"\nloss 10 epoch cuoi: {last[0]:.4f} -> {last[-1]:.4f} "
      f"(giam {100*(last[0]-last[-1])/last[0]:+.1f}%)")


nguong  Dice
  0.20  0.755
  0.25  0.784
  0.30  0.809
  0.35  0.832
  0.40  0.852
  0.45  0.870
  0.50  0.883  <-- 0.5 mac dinh
  0.55  0.893
  0.60  0.897
  0.65  0.894
  0.70  0.886
  0.75  0.869
  0.80  0.839
=> tot nhat 0.897 tai nguong 0.60

loi tong 4.87% | o BIEN 31.67% | trong LONG 3.70%
ty trong loi nam o bien: 27.2%

loss 10 epoch cuoi: 0.1197 -> 0.1051 (giam +12.2%)


### 4. Ma trận P0–P3 + bảng ablation M8 (§7, §8 — bước 7)

P3 tự bỏ qua nếu chưa có OOF softmax. **M8-A/M8-B không nằm trong ma trận P**
(P0/P1 dùng D0+H0) nên phải dựng riêng dưới scaffold `S0-D1-H1`.

In [9]:
# ---- CACHE HINH HOC: bo viec dung lai marching-cubes/EDT cho moi run -------------
# Muc 4 chay 5 run x 50 ca = 250 lan dung hinh hoc, nhung be mat/phap tuyen/occupancy
# GIONG HET NHAU giua cac run chi khac KENH. Cache phan dung chung do.
# KHOA CACHE bao gom MOI yeu to anh huong hinh hoc - thieu mot cai la dung nham cache
# cu ma khong bao loi (loai bug im lang).
import hashlib

GEOM_CACHE = f"{BSC_ROOT}/geom_cache"
os.makedirs(GEOM_CACHE, exist_ok=True)
_orig_build_case = getattr(mvp, "_orig_build_case", mvp.build_case)
mvp._orig_build_case = _orig_build_case

def _geom_key(run, cid, cfg, direction, js, jt):
    parts = [cid, run.surface, run.domain, direction, f"{js:.4f}", f"{jt:.4f}",
             f"k{cfg.k}", f"{cfg.d_min:.3f}", f"{cfg.d_max:.3f}",
             f"{cfg.smooth_mm:.3f}", f"cls{run.cls}"]
    return hashlib.sha1("|".join(parts).encode()).hexdigest()[:16]

def _cached_build_case(run, src, cid, cfg=None, atlas=None, direction="normal",
                       jitter_s_mm=0.0, jitter_theta_deg=0.0, seed=0):
    cfg = cfg or RayConfig()
    js, jt = jitter_s_mm, jitter_theta_deg
    if run.surface == "S1" and js == 0.0 and jt == 0.0:
        js, jt = 0.5, 10.0                       # phai khop mac dinh cua ban goc

    path = f"{GEOM_CACHE}/{_geom_key(run, cid, cfg, direction, js, jt)}.npz"
    sp = src.spacing(cid)

    if os.path.exists(path):
        z = np.load(path)
        verts, dirs = z["verts"], z["dirs"]
        occ = np.unpackbits(z["occ"], axis=1, count=cfg.k).astype(np.uint8)
    else:
        out = _orig_build_case(run, src, cid, cfg, atlas, direction, js, jt, seed)
        if out is None:
            return None
        _, occ, _, verts, dirs = out
        np.savez_compressed(path, verts=verts, dirs=dirs,
                            occ=np.packbits(occ.astype(bool), axis=1),
                            dom=np.ones(len(verts), bool))

    mri = src.mri(cid).astype(np.float32)
    mri = (mri - float(mri.mean())) / (float(mri.std()) + 1e-6)
    pool = {"mri": mri}
    if "grad" in run.channels:
        pool["grad"] = mvp.model.mri_gradient(mri, sp)
    if "sdf" in run.channels:
        bone = src.bone_pred(cid) if run.surface == "S2" else src.bone_gt(cid)
        pool["sdf"] = core.signed_distance(bone, sp, smooth_mm=cfg.smooth_mm)
    if "prob" in run.channels:
        pool["prob"] = src.prob(cid).astype(np.float32)

    X = core.sample_rays({c: pool[c] for c in run.channels}, verts, dirs, cfg, sp)
    pres = core.ray_stats(occ, cfg)["presence"].astype(np.uint8)
    return X, occ, pres, verts, dirs

mvp.build_case = _cached_build_case
print(f"Cache hinh hoc BAT -> {GEOM_CACHE}")
print("Go cache:  mvp.build_case = mvp._orig_build_case")


Cache hinh hoc BAT -> /content/drive/MyDrive/bsc/geom_cache
Go cache:  mvp.build_case = mvp._orig_build_case


In [10]:
RUNS, EPOCHS = {}, 30

def run_and_log(run, tag):
    res = mvp.train_run(run, SRC, train_ids, val_ids, cfg, atlas=ATLAS,
                        rays_per_case=RAYS, epochs=EPOCHS, device="cuda")
    RUNS[tag] = res
    p = f" | presence F1 {res.presence['f1']:.3f} absent-recall {res.presence['absent_recall']:.3f}" \
        if res.presence else " | (H0: khong presence)"
    print(f"{tag:<7}{run.experiment_id}")
    print(f"        {run.channels} {run.domain}/{run.heads} | occ-Dice {res.val_occ_dice:.3f}{p}")
    return res

for alias in ("P0", "P1", "P2", "P3"):
    kw = {"prob_source": X.DEFAULT_PROB_SOURCE} if alias == "P3" else {}
    run = X.from_plan(alias, CLS, seed=1, **kw)
    if "prob" in run.channels and not glob.glob(f"{PROB_DIR}/{CLS}/*.nii.gz"):
        print(f"{alias}: bo qua - chua co OOF softmax"); continue
    run_and_log(run, alias)

# --- M8: scaffold S0-D1-H1, chi doi kenh (§8) ---
for role in ("M8-A", "M8-B"):
    inp = X.M8_INPUTS[role]
    run = X.RunConfig(cls=CLS, surface="S0", domain="D1", inputs=inp, heads="H1", seed=1)
    run_and_log(run, role)

print(f"{'M8':<7}{'run':<8}{'kenh':<20}{'occ-Dice':>10}{'presence F1':>13}")
for tag, r in RUNS.items():
    role = r.run.m8_role
    if not role: continue
    f1 = f"{r.presence['f1']:.3f}" if r.presence else "-"
    print(f"{role:<7}{tag:<8}{str(r.run.channels):<20}{r.val_occ_dice:>10.3f}{f1:>13}")

# Dong gop coarse prior: moc DUNG la M8-A (I0=mri) vs M8-D (I3=mri+prob) - khac dung
# MOT kenh. So M8-C (I2=mri+sdf) voi M8-D se doi HAI kenh cung luc (§9 canh bao).
if "M8-A" in RUNS and "P3" in RUNS:
    d = RUNS["P3"].val_occ_dice - RUNS["M8-A"].val_occ_dice
    print(f"Dong gop coarse ResEnc prior (M8-A -> M8-D): {d:+.3f} occ-Dice")
    print("Lon => mo hinh chu yeu HIEU CHINH ResEnc, khong tu doc MRI (§3.5 M8).")

P0     MVP_FC_S0_D0_I0_H0_v1_Fold0_Seed1
        ('mri',) D0/H0 | occ-Dice 0.833 | (H0: khong presence)
P1     MVP_FC_S0_D0_I1_H0_v1_Fold0_Seed1
        ('mri', 'grad') D0/H0 | occ-Dice 0.849 | (H0: khong presence)
P2     MVP_FC_S0_D1_I2_H1_v1_Fold0_Seed1
        ('mri', 'sdf') D1/H1 | occ-Dice 0.825 | presence F1 0.884 absent-recall 0.916
P3: bo qua - chua co OOF softmax
M8-A   MVP_FC_S0_D1_I0_H1_v1_Fold0_Seed1
        ('mri',) D1/H1 | occ-Dice 0.821 | presence F1 0.880 absent-recall 0.907
M8-B   MVP_FC_S0_D1_I1_H1_v1_Fold0_Seed1
        ('mri', 'grad') D1/H1 | occ-Dice 0.839 | presence F1 0.890 absent-recall 0.927
M8     run     kenh                  occ-Dice  presence F1
M8-C   P2      ('mri', 'sdf')           0.825        0.884
M8-A   M8-A    ('mri',)                 0.821        0.880
M8-B   M8-B    ('mri', 'grad')          0.839        0.890


### 5. M6 — negative control hướng tia (§3.5, bước 9)

**Test phản bác chính.** Hướng tùy ý tốt ngang pháp tuyến ⇒ lợi ích không đến từ hệ tọa độ.

In [11]:
BEST = "P2" if "P2" in RUNS else next(iter(RUNS))
base_run = RUNS[BEST].run
m6 = {}
for mode in ("normal", "axial", "tangent", "random"):
    res = mvp.train_run(base_run, SRC, train_ids, val_ids, cfg, atlas=ATLAS,
                        rays_per_case=RAYS, epochs=EPOCHS, direction=mode, device="cuda")
    m6[mode] = {"occ_dice": res.val_occ_dice,
                "presence_f1": res.presence["f1"] if res.presence else None}
    print(f"{mode:<9} occ-Dice {res.val_occ_dice:.3f}")

ctrl = max(m6[k]["occ_dice"] for k in ("axial", "tangent", "random"))
print(f"M6: normal {m6['normal']['occ_dice']:.3f} vs doi chung tot nhat {ctrl:.3f}")
print("CONG: normal phai HON HAN. Neu khong => phai bao cao trung thuc dieu do.")

normal    occ-Dice 0.825
axial     occ-Dice 0.856
tangent   occ-Dice 0.837
random    occ-Dice 0.798
M6: normal 0.825 vs doi chung tot nhat 0.856
CONG: normal phai HON HAN. Neu khong => phai bao cao trung thuc dieu do.


In [12]:
# M6 do DUNG theo §3.5: metric BIEN o khong gian voxel, khong phai occ-Dice khong gian tia.
m6_fix, m6_nets = {}, {}
for mode in ("normal", "axial", "tangent", "random"):
    r = mvp.train_run(base_run, SRC, train_ids, val_ids, cfg, atlas=ATLAS,
                      rays_per_case=RAYS, epochs=EPOCHS, direction=mode, device="cuda")
    m6_nets[mode] = r.net
    Xa, oa, _ = mvp.build_dataset(base_run, SRC, train_ids[:3], cfg, atlas=ATLAS,
                                  rays_per_case=500, direction=mode)
    m6_fix[mode] = {"occ_dice_rayspace": r.val_occ_dice, "pos_frac": float(oa.mean())}

print(f"{'huong':<9}{'ty le duong':>12}{'occ-Dice(tia)':>15}   <- xac nhan confound")
for k, v in m6_fix.items():
    print(f"{k:<9}{v['pos_frac']:>11.1%}{v['occ_dice_rayspace']:>15.3f}")

print(f"\n{'huong':<9}{'loi bien vung mong':>20}{'so ca':>8}   <- PHEP DO DUNG")
for mode, net in m6_nets.items():
    errs = []
    for cid in val_ids[:5]:
        r = mvp.evaluate_case(base_run, net, SRC, cid, cfg, ATLAS, device="cuda")
        if r: errs.append(r["thin_ray"]["thin_mean_err_mm"])
    m6_fix[mode]["thin_err_mm"] = float(np.nanmean(errs))
    print(f"{mode:<9}{np.nanmean(errs):>19.4f}mm{len(errs):>8}")

best_ctrl = min(m6_fix[k]["thin_err_mm"] for k in ("axial","tangent","random"))
print(f"\nnormal {m6_fix['normal']['thin_err_mm']:.4f}mm vs doi chung tot nhat {best_ctrl:.4f}mm")
print("CONG: normal phai THAP HON (loi it hon) ro ret.")


huong     ty le duong  occ-Dice(tia)   <- xac nhan confound
normal         21.2%          0.825
axial          33.8%          0.856
tangent        35.7%          0.837
random         24.5%          0.797

huong      loi bien vung mong   so ca   <- PHEP DO DUNG
normal                1.9394mm       5
axial                 3.7639mm       5
tangent               2.9498mm       5
random                2.2026mm       5

normal 1.9394mm vs doi chung tot nhat 2.2026mm
CONG: normal phai THAP HON (loi it hon) ro ret.


In [13]:
# M6 - thong ke ghep cap theo §2.4. Dung DU 10 ca val.
per_dir = {}
for mode, net in m6_nets.items():
    v = {}
    for cid in val_ids:
        r = mvp.evaluate_case(base_run, net, SRC, cid, cfg, ATLAS, device="cuda")
        if r:
            v[cid] = r["thin_ray"]["thin_mean_err_mm"]
    per_dir[mode] = v

common = sorted(set.intersection(*[set(v) for v in per_dir.values()]))
nrm = np.array([per_dir["normal"][c] for c in common], float)
print(f"n = {len(common)} ca ghep cap | normal {nrm.mean():.4f}mm")
print()
print(f"{'doi chung':<10}{'loi TB':>10}{'normal tot hon':>16}{'boot 95% CI':>22}{'so ca':>9}")

m6_stats = {}
for mode in ("axial", "tangent", "random"):
    ctl = np.array([per_dir[mode][c] for c in common], float)
    b = metrics.paired_bootstrap(nrm, ctl)          # ctl - nrm > 0 = normal tot hon
    lo, hi = b["ci_low"], b["ci_high"]
    ci = f"[{lo:+.4f}, {hi:+.4f}]"
    m6_stats[mode] = {"ctl_mm": float(ctl.mean()), "delta_mm": b["delta_mean"],
                      "ci": (lo, hi), "n_better": b["n_better"], "n": b["n"],
                      "sig": bool(lo > 0)}
    print(f"{mode:<10}{ctl.mean():>10.4f}{b['delta_mean']:>+16.4f}{ci:>22}"
          f"{b['n_better']:>6}/{b['n']}")

all_sig = all(v["sig"] for v in m6_stats.values())
print()
print(f"M6 (mau so vung mong): {'DAT' if all_sig else 'CHUA DAT'}")
print("CONG: MOI doi chung phai co CI duoi > 0.")
if not all_sig:
    print(f"  chua chac thang: {[k for k, v in m6_stats.items() if not v['sig']]}")


n = 10 ca ghep cap | normal 1.9865mm

doi chung     loi TB  normal tot hon           boot 95% CI    so ca
axial         3.6755         +1.6889    [+1.3853, +1.9272]    10/10
tangent       2.9681         +0.9816    [+0.6766, +1.2910]    10/10
random        2.3160         +0.3295    [+0.2146, +0.4627]     9/10

M6 (mau so vung mong): DAT
CONG: MOI doi chung phai co CI duoi > 0.


In [14]:
# M6 - thong ke ghep cap theo §2.4. Dung DU 10 ca val.
per_dir = {}
for mode, net in m6_nets.items():
    v = {}
    for cid in val_ids:
        r = mvp.evaluate_case(base_run, net, SRC, cid, cfg, ATLAS, device="cuda")
        if r:
            v[cid] = r["thin_ray"]["thin_mean_err_mm"]
    per_dir[mode] = v

common = sorted(set.intersection(*[set(v) for v in per_dir.values()]))
nrm = np.array([per_dir["normal"][c] for c in common], float)
print(f"n = {len(common)} ca ghep cap | normal {nrm.mean():.4f}mm")
print()
print(f"{'doi chung':<10}{'loi TB':>10}{'normal tot hon':>16}{'boot 95% CI':>22}{'so ca':>9}")

m6_stats = {}
for mode in ("axial", "tangent", "random"):
    ctl = np.array([per_dir[mode][c] for c in common], float)
    b = metrics.paired_bootstrap(nrm, ctl)          # ctl - nrm > 0 = normal tot hon
    lo, hi = b["ci_low"], b["ci_high"]
    ci = f"[{lo:+.4f}, {hi:+.4f}]"
    m6_stats[mode] = {"ctl_mm": float(ctl.mean()), "delta_mm": b["delta_mean"],
                      "ci": (lo, hi), "n_better": b["n_better"], "n": b["n"],
                      "sig": bool(lo > 0)}
    print(f"{mode:<10}{ctl.mean():>10.4f}{b['delta_mean']:>+16.4f}{ci:>22}"
          f"{b['n_better']:>6}/{b['n']}")

all_sig = all(v["sig"] for v in m6_stats.values())
print()
print(f"M6 (mau so vung mong): {'DAT' if all_sig else 'CHUA DAT'}")
print("CONG: MOI doi chung phai co CI duoi > 0.")
if not all_sig:
    print(f"  chua chac thang: {[k for k, v in m6_stats.items() if not v['sig']]}")


n = 10 ca ghep cap | normal 1.9865mm

doi chung     loi TB  normal tot hon           boot 95% CI    so ca
axial         3.6755         +1.6889    [+1.3853, +1.9272]    10/10
tangent       2.9681         +0.9816    [+0.6766, +1.2910]    10/10
random        2.3160         +0.3295    [+0.2146, +0.4627]     9/10

M6 (mau so vung mong): DAT
CONG: MOI doi chung phai co CI duoi > 0.


### 6. M7 jitter + P4/P5 (§3.5, bước 10 & 12)

§4.9 đặt mốc: giữ **60–70%** hiệu năng khi chuyển sang bề mặt xương dự đoán.

In [15]:
m7 = {}
for ds, dth in [(0.0, 0), (0.25, 5), (0.5, 10), (1.0, 15)]:
    Xb, ob, pb = mvp.build_dataset(base_run, SRC, val_ids, cfg, atlas=ATLAS,
                                   rays_per_case=RAYS, seed=1000,
                                   jitter_s_mm=ds, jitter_theta_deg=dth)
    op, pp = M.predict_rays(RUNS[BEST].net, Xb, device="cuda")
    d = float(2*((op>0.5) & ob.astype(bool)).sum()/((op>0.5).sum()+ob.sum()+1e-8))
    m7[f"{ds}mm/{dth}deg"] = d
    print(f"jitter {ds:.2f}mm/{dth:>2}deg  occ-Dice {d:.3f}")
print("Suy giam PHAI tu tu. Sup dot ngot => he toa do gion, xem lai truoc P5.\n")

BEST_I = base_run.inputs
for alias in ("P4", "P5"):
    kw = {"prob_source": X.DEFAULT_PROB_SOURCE} if BEST_I == "I3" else {}
    run = X.from_plan(alias, CLS, inputs=BEST_I, seed=1, **kw)
    if alias == "P5" and any(not SRC.has_pred(c) for c in train_ids + val_ids):
        print("P5: bo qua - co ca thieu prediction xuong"); continue
    run_and_log(run, alias)

if "P5" in RUNS:
    print(f"P5 giu {RUNS['P5'].val_occ_dice / RUNS[BEST].val_occ_dice:.0%} hieu nang "
          f"so voi xuong GT (§4.9 moc 60-70%)")

jitter 0.00mm/ 0deg  occ-Dice 0.825
jitter 0.25mm/ 5deg  occ-Dice 0.735
jitter 0.50mm/10deg  occ-Dice 0.623
jitter 1.00mm/15deg  occ-Dice 0.492
Suy giam PHAI tu tu. Sup dot ngot => he toa do gion, xem lai truoc P5.

P4     MVP_FC_S1_D1_I2_H1_v1_Fold0_Seed1
        ('mri', 'sdf') D1/H1 | occ-Dice 0.805 | presence F1 0.893 absent-recall 0.865
P5     MVP_FC_S2_D1_I2_H1_v1_Fold0_Seed1
        ('mri', 'sdf') D1/H1 | occ-Dice 0.793 | presence F1 0.891 absent-recall 0.891
P5 giu 96% hieu nang so voi xuong GT (§4.9 moc 60-70%)


### 7. Đánh giá §3.7 (mẫu số vùng mỏng) + cổng Go/No-Go — bước 13

In [ ]:
EVAL_CKPT = f"{BSC_ROOT}/runs/mvp_eval_{CLS}_{BEST}.jsonl"
have = ({json.loads(l)["case"] for l in open(EVAL_CKPT)}
        if os.path.exists(EVAL_CKPT) else set())
with open(EVAL_CKPT, "a") as fh:
    for cid in tqdm(val_ids, desc="eval"):
        if cid in have: continue
        r = mvp.evaluate_case(base_run, RUNS[BEST].net, SRC, cid, cfg, ATLAS, device="cuda")
        if r:
            fh.write(json.dumps(r) + chr(10)); fh.flush()

rows = [json.loads(l) for l in open(EVAL_CKPT)]
g = mvp.summarize_gate37(rows)
print(f"n = {g['n']} ca val (out-of-fold)")
print(f"loi bien vung mong  baseline {g['thin_err_baseline_mm']:.4f}mm -> "
      f"ray {g['thin_err_ray_mm']:.4f}mm")
print(f"cai thien tuong doi {g['rel_improve']:+.1%}  (muc tieu §3.7: >= +10%)")
print(f"cai thien tuyet doi {g['abs_improve_mm']:+.4f}mm  CI {g['ci']}")
print(f"so ca tot hon {g['n_better']}/{g['n']}")
if "presence_f1" in g:
    print(f"presence F1 {g['presence_f1']:.3f} | absent recall {g['absent_recall']:.3f}")

# M6 gate DUNG metric bien (m6_stats tu cell M6 paired), KHONG dung occ-Dice cu (review §3.4)
m6_pass = bool("m6_stats" in dir() and all(v["sig"] for v in m6_stats.values()))
gate = {"1_thin_rel_improve": g.get("rel_improve"),
        "1_pass_10pct": g.get("pass_10pct"),
        "1_ci_low_positive": g.get("ci_low_positive"),
        "5_m6_normal_beats_controls": m6_pass,          # <- doc m6_stats (boundary metric)
        "m5_passed": bool(m5_dice > 0.90),
        "6_p5_retained": (RUNS["P5"].val_occ_dice / RUNS[BEST].val_occ_dice
                          if "P5" in RUNS else None)}
print("\nLUU Y: day la aggregate flow CU (exploratory). Ket qua chinh thuc dung Phase A")
print("(A1/A2/A3) tu checkpoint khoa hash. m5_passed/P5 con caveat - xem doc §14.")
print(json.dumps(gate, indent=2, ensure_ascii=False))

registry = []
for tag, r in RUNS.items():
    rec = r.run.to_registry(dataset_revision="Dataset001_KneeOA",
                            extra={"tag": tag, "val_occ_dice": r.val_occ_dice,
                                   "presence": r.presence, "n_train_rays": r.n_train_rays,
                                   "n_train_cases": len(train_ids), "n_val_cases": len(val_ids),
                                   "epochs": EPOCHS, "rays_per_case": RAYS,
                                   "ray_k": cfg.k, "ray_d_min": cfg.d_min,
                                   "ray_d_max": cfg.d_max, "smooth_mm": cfg.smooth_mm,
                                   "atlas_n_cases": len(ATLAS.case_ids), "spacing": list(SP)})
    registry.append(rec)

out = {"class": CLS, "registry": registry, "m6": m6,
       "m6_stats": (m6_stats if "m6_stats" in dir() else None), "m7": m7,
       "gate37": {**g, **gate}, "m5_occ_dice": m5_dice}
path = f"{BSC_ROOT}/runs/MVP_stage1_{CLS}.json"
json.dump(out, open(path, "w"), indent=2, ensure_ascii=False, default=str)
print("Da ghi", path)

### 8. `med_tib_cart` — lop stress-test (plan §3.1, §6 buoc 11)

**Muc DUY NHAT con thieu cua Stage 1.** Chay de dong ho so, khong phai de "cuu" gia thuyet.

Cau hinh **y het canonical P2 femoral** (40 train / 10 val, 20k tia, 30 epoch, lr 3e-4,
seed 1) => chi doi DUNG MOT yeu to la **lop** (QD4, §3.2).

Vi sao van phai chay du ket qua femoral am tinh: M0 cho med_tib prize **0.0885mm**
(cao hon femoral 0.0582mm **52%**) va thin mass **33.5%** vs 21.4% => khong hien nhien.

**Chay:** cell 2 (config) -> 8a -> 8b. Khong can chay cell 4/6/8/9 truoc.
**GPU:** T4 la du (nut that la CPU dung hinh hoc). A100 lang phi.


In [ ]:
# 8a - med_tib: split + atlas + train canonical P2 -----------------------------
# Tu chua: chi can cell 2 da chay. KHONG dung/ghi de bien cua phien femoral.
import hashlib
from bsc import model as M

CLS_MT = "med_tib_cart"
N_TRAIN, N_VAL, RAYS_MT = 40, 10, 20000          # Y HET canonical femoral
EPOCHS_MT, LR_MT = 30, 3e-4

# --- split (dung lai neu phien nay da co, khong thi dung lai tu dia) ---------
if "cases_all" not in dir():
    cases_all = sorted(os.path.basename(p)[:-len("_0000.nii.gz")]
                       for p in glob.glob(f"{RAW}/imagesTr/*_0000.nii.gz"))
    cases_all = [c for c in cases_all if c.startswith("oaizib_")]
if "SPLITS" not in dir():
    SPLITS = f"{BSC_ROOT}/splits/splits_zib_v1_fixed.json"

fold_of = json.load(open(SPLITS))["fold_of"]
zib = [c for c in cases_all if c in fold_of]
tr_mt = [c for c in zib if fold_of[c] != 0][:N_TRAIN]
va_mt = [c for c in zib if fold_of[c] == 0][:N_VAL]
print(f"train {len(tr_mt)} | val {len(va_mt)}")

CV_DIR_MT = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_150epochs*/",
                      recursive=True)[0]
def _bl_mt(cid):
    p = glob.glob(f"{CV_DIR_MT}/fold_*/validation/{cid}.nii.gz")
    return p[0] if p else None

SRC_MT = mvp.NiftiCaseSource(RAW, CLS_MT, CART[CLS_MT], BONE[CLS_MT], _bl_mt, PROB_DIR)

# --- atlas fold-0 cho med_tib (nap neu co, khong thi xay ~15 phut CPU) -------
P_MT = f"{BSC_ROOT}/atlas/atlas_{CLS_MT}_fold0.npz"
if os.path.exists(P_MT):
    z = np.load(P_MT, allow_pickle=True)
    ATLAS_MT = atlas_mod.ArticularAtlas(z["prob"], int(z["n_bins"]), float(z["lo"]),
                                        float(z["hi"]), tuple(z["case_ids"]), 0, None)
    print(f"Nap atlas med_tib: {len(ATLAS_MT.case_ids)} ca")
else:
    def _al_mt(cid): return SRC_MT.mri(cid), SRC_MT.bone_gt(cid), SRC_MT.cart_gt(cid)
    ATLAS_MT = atlas_mod.build_articular_atlas(tr_mt, _al_mt, SP, cfg,
                                              n_bins=24, min_count=3, fold=0)
    np.savez_compressed(P_MT, prob=ATLAS_MT.prob, n_bins=ATLAS_MT.n_bins,
                        lo=ATLAS_MT.lo, hi=ATLAS_MT.hi,
                        case_ids=np.array(ATLAS_MT.case_ids))
    print(f"Xay atlas med_tib tu {len(ATLAS_MT.case_ids)} ca -> {P_MT}")

atlas_mod.assert_no_leak(ATLAS_MT, va_mt)        # 2.2 - phai KHONG throw
print(f"assert_no_leak OK | phu {np.isfinite(ATLAS_MT.prob).mean():.1%}, "
      f"P(khop) TB {np.nanmean(ATLAS_MT.prob):.3f}")

# SANITY: atlas med_tib phai KHAC atlas femoral (bat nham file / nham lop)
_pf = f"{BSC_ROOT}/atlas/atlas_femoral_cart_fold0.npz"
if os.path.exists(_pf):
    _pfem = np.load(_pf, allow_pickle=True)["prob"]
    _same = (_pfem.shape == ATLAS_MT.prob.shape and np.array_equal(
        np.nan_to_num(_pfem, nan=-1), np.nan_to_num(ATLAS_MT.prob, nan=-1)))
    assert not _same, "atlas med_tib TRUNG femoral - kiem lai file/lop!"
    print("sanity: khac atlas femoral OK")

# --- train canonical P2 (chi doi lop) ---------------------------------------
run_mt = X.from_plan("P2", CLS_MT, seed=1)
if getattr(mvp, "_orig_build_case", None) is not None:
    mvp.build_case = mvp._orig_build_case     # go cache neu phien truoc bat

ah = hashlib.sha256(ATLAS_MT.prob.tobytes()
                    + str(sorted(ATLAS_MT.case_ids)).encode()).hexdigest()[:16]
sh = hashlib.sha256(json.dumps({"train": sorted(tr_mt), "val": sorted(va_mt)},
                               sort_keys=True).encode()).hexdigest()[:16]

res_mt = mvp.train_run(run_mt, SRC_MT, tr_mt, va_mt, cfg, atlas=ATLAS_MT,
                       rays_per_case=RAYS_MT, epochs=EPOCHS_MT, lr=LR_MT, device="cuda")

run_dir_mt = mvp.save_run(res_mt, BSC_ROOT, cfg, epochs=EPOCHS_MT, lr=LR_MT,
                          rays_per_case=RAYS_MT, atlas_hash=ah, split_hash=sh,
                          dataset_revision="Dataset001_KneeOA", git_cwd=REPO_DIR)
man_mt = mvp.run_manifest(run_dir_mt)
print(chr(10) + "run_dir =", run_dir_mt)
print(f"experiment_id: {man_mt['experiment_id']}")
print(f"ckpt {man_mt['checkpoint_sha256']} | git {man_mt['git_commit'][:8]} "
      f"| cfg {man_mt['config_hash']} | atlas {ah} | split {sh}")
print(f"val occ-Dice: {man_mt['val_occ_dice']:.3f}")


In [ ]:
# 8b - med_tib: danh gia 3.7 tu MODEL RELOAD + phan quyet ---------------------
net_mt = mvp.load_run_model(run_dir_mt, device="cuda")   # KHONG dung res_mt.net trong RAM

EVAL_MT = f"{run_dir_mt}/eval_gate37.jsonl"
open(EVAL_MT, "w").close()
rows_mt = []
for cid in tqdm(va_mt, desc="gate37 med_tib"):
    r = mvp.evaluate_case(run_mt, net_mt, SRC_MT, cid, cfg, ATLAS_MT, device="cuda")
    if r:
        rows_mt.append(r)
        open(EVAL_MT, "a").write(json.dumps(r) + chr(10))

g = mvp.summarize_gate37(rows_mt)
print(chr(10) + "=" * 66)
print(f"med_tib_cart  n={g['n']}")
print(f"  thin-region err:  B0 {g['thin_err_baseline_mm']:.4f}mm "
      f"->  ray {g['thin_err_ray_mm']:.4f}mm   ({g['rel_improve']:+.1%})")
print(f"  so ca tot hon B0: {g['n_better']}/{g['n']}")
if g.get('ci'):
    print(f"  paired bootstrap CI95 (B0 - ray): [{g['ci'][0]:+.4f}, {g['ci'][1]:+.4f}] mm"
          f"   CI duong? {g['ci_low_positive']}")
if "presence_f1" in g:
    print(f"  presence F1 {g['presence_f1']:.3f} | absent recall {g['absent_recall']:.3f}")

# doi chieu femoral (canonical da khoa) - chi de nhin, khong phai test thong ke
FEM = {"thin_ray": 2.1881, "thin_b0": 0.5517, "n_better": 0, "n": 10}
print(chr(10) + f"{'lop':<14}{'B0 (mm)':>10}{'ray (mm)':>11}{'ty le':>9}{'tot hon':>10}")
print(f"{'femoral_cart':<14}{FEM['thin_b0']:>10.4f}{FEM['thin_ray']:>11.4f}"
      f"{FEM['thin_ray']/FEM['thin_b0']:>8.1f}x{FEM['n_better']:>7}/{FEM['n']}")
print(f"{'med_tib_cart':<14}{g['thin_err_baseline_mm']:>10.4f}{g['thin_err_ray_mm']:>11.4f}"
      f"{g['thin_err_ray_mm']/g['thin_err_baseline_mm']:>8.1f}x"
      f"{g['n_better']:>7}/{g['n']}")

# CONG 3.7: ray phai TOT HON B0 (thin-region err thap hon) tren da so ca
PASS = (g["thin_err_ray_mm"] < g["thin_err_baseline_mm"]) and (g["n_better"] > g["n"] / 2)
print(chr(10) + ("=> DAT 3.7 - KHONG dong Stage 1, phai dieu tra vi sao med_tib khac femoral"
                 if PASS else
                 "=> TRUOT 3.7 tren CA HAI lop => Stage 1 KET LUAN AM TINH, dong."))

json.dump({"gate37": g, "pass": bool(PASS),
           "checkpoint_sha256": man_mt["checkpoint_sha256"],
           "experiment_id": man_mt["experiment_id"],
           "git_commit": man_mt["git_commit"],
           "compare_femoral": FEM},
          open(f"{run_dir_mt}/gate37_summary.json", "w"), indent=2)
print("khoa tai", run_dir_mt)
